# PXIe-4080 Gain Error Analysis

This notebook parses TDMS files and plots gain error for the **10V** and **300V** ranges of the National Instruments PXIe-4080 across all DUTs, alongside chamber temperature and humidity.

Run the cells top-to-bottom. Edit the configuration cell to point at your data and tweak parameters inline.

## 1. Imports

In [ ]:
%matplotlib inline

import os
import re
from datetime import datetime

from nptdms import TdmsFile
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

## 2. Helper functions

In [ ]:
def compute_gain_error_ppm(test_points, dut_meas):
    # Gain error in ppm comes from slope deviation from unity.
    X = np.asarray(test_points).reshape(-1, 1)
    y = np.asarray(dut_meas)
    reg = LinearRegression().fit(X, y)
    slope = reg.coef_[0]
    return (slope - 1) * 1_000_000


def serial_sort_key(serial):
    normalized = str(serial).strip()
    if normalized.lower().startswith('0x'):
        normalized = normalized[2:]
    try:
        return (0, int(normalized, 16), str(serial))
    except ValueError:
        # Fallback keeps stable lexical ordering for unexpected serial formats.
        return (1, str(serial))


def get_first_available_channel(group, channel_names):
    for channel_name in channel_names:
        if channel_name in group:
            return group[channel_name][:]
    joined = ", ".join(channel_names)
    raise KeyError(f"No channel found from candidates: {joined}")


def parse_filename(filename):
    name = filename[:-5].strip()
    # Normalize extra spaces and AM/PM spacing
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'\s+(AM|PM)$', r'_\1', name)
    name = name.replace('RH_Set_ ', 'RH_Set_')

    pattern = re.compile(
        r'^(?P<serial>[^_]+)_Temp_Set_(?P<temp>[^_]+)_RH_Set_(?P<rh>\d+)_(?P<month>\d+)_(?P<day>\d+)_(?P<year>\d+)_(?P<hour>\d+)_(?P<minute>\d+)_(?P<second>\d+)_(?P<am_pm>AM|PM)$'
    )
    match = pattern.match(name)
    if not match:
        raise ValueError(f"Unable to parse filename: {filename}")

    serial = match.group('serial')
    temp_set = match.group('temp')
    rh_set = match.group('rh')
    month = int(match.group('month'))
    day = int(match.group('day'))
    year = int(match.group('year'))
    hour = int(match.group('hour'))
    minute = int(match.group('minute'))
    second = int(match.group('second'))
    am_pm = match.group('am_pm')

    dt = datetime(year, month, day, hour, minute, second)
    if am_pm == 'PM' and hour != 12:
        dt = dt.replace(hour=hour + 12)
    elif am_pm == 'AM' and hour == 12:
        dt = dt.replace(hour=0)
    return serial, temp_set, rh_set, dt

## 3. Configuration

Set the root path that contains one subfolder per DUT serial number. Adjust `GAIN_ERROR_LIMIT_PPM` to change the outlier cutoff.

In [ ]:
ROOT_PATH = r"C:\temp\EnvironmentalTesting\Round2Data"
GAIN_ERROR_LIMIT_PPM = 100
OUTPUT_IMAGE = "gain_error_plot.png"

## 4. Load TDMS data

Walks each serial folder, computes gain error for the 10V and 300V ranges, and drops any point outside +/- `GAIN_ERROR_LIMIT_PPM`.

In [ ]:
data = []

if not os.path.exists(ROOT_PATH):
    raise FileNotFoundError(f"Path '{ROOT_PATH}' does not exist. Please verify the directory path.")

for subdir in os.listdir(ROOT_PATH):
    serial_path = os.path.join(ROOT_PATH, subdir)
    if not os.path.isdir(serial_path):
        continue
    for file in os.listdir(serial_path):
        if not file.endswith('.tdms'):
            continue
        filepath = os.path.join(serial_path, file)
        try:
            parsed_serial, temp_set, rh_set, timestamp = parse_filename(file)
            tdms_file = TdmsFile.read(filepath)
            group = tdms_file.groups()[0]
            chamber_temp = get_first_available_channel(
                group,
                ['Chamber Temperature 300V', 'Chamber Temperature']
            )
            chamber_rh = get_first_available_channel(
                group,
                ['Chamber RH% 300V', 'Chamber RH%']
            )
            avg_temp = np.mean(chamber_temp)
            avg_rh = np.mean(chamber_rh)

            ranges = [
                ('10V', ['Test Points 10V'], ['DUT Measurements 10V']),
                ('300V', ['Test Points 300V', 'Test Points'], ['DUT Measurements 300V', 'DUT Measurements'])
            ]

            for range_name, test_candidates, dut_candidates in ranges:
                test_col = next((col for col in test_candidates if col in group), None)
                dut_col = next((col for col in dut_candidates if col in group), None)
                if test_col is None or dut_col is None:
                    continue

                test_points = group[test_col][:]
                dut_meas = group[dut_col][:]
                gain_error = compute_gain_error_ppm(test_points, dut_meas)

                # Omit outliers outside the valid gain error window.
                if abs(gain_error) > GAIN_ERROR_LIMIT_PPM:
                    print(
                        f"Omitting {range_name} point from {file}: "
                        f"gain error {gain_error:.2f} ppm exceeds +/-{GAIN_ERROR_LIMIT_PPM} ppm"
                    )
                    continue

                data.append({
                    'timestamp': timestamp,
                    'serial': parsed_serial,
                    'range': range_name,
                    'gain_error': gain_error,
                    'avg_temp': avg_temp,
                    'avg_rh': avg_rh
                })
        except Exception as e:
            print(f"Error processing {filepath}: {e}")

df = pd.DataFrame(data)
if df.empty:
    print("No data found.")
else:
    df = df.sort_values(['serial', 'timestamp'])
df.head()

## 5. Plot gain error and chamber conditions

Each serial gets a unique color shared by its 10V (dashed, square markers) and 300V (solid, round markers) traces.

In [ ]:
if df.empty:
    raise ValueError("No data to plot. Run the data-loading cell first.")

fig, ax1 = plt.subplots(figsize=(14, 8))

# Gain error for each serial in ascending order.
serials = sorted(df['serial'].unique(), key=serial_sort_key)

# Assign a unique color per serial so 10V and 300V share the serial's color.
colormap = plt.get_cmap('tab20' if len(serials) > 10 else 'tab10')
serial_colors = {ser: colormap(idx % colormap.N) for idx, ser in enumerate(serials)}

for ser in serials:
    ser_df = df[df['serial'] == ser]
    for range_name, linestyle, marker in [('10V', '--', 's'), ('300V', '-', 'o')]:
        range_df = ser_df[ser_df['range'] == range_name]
        if range_df.empty:
            continue
        ax1.plot(
            range_df['timestamp'],
            range_df['gain_error'],
            label=f'{ser} {range_name}',
            color=serial_colors[ser],
            linestyle=linestyle,
            marker=marker
        )

ax1.set_xlabel('Time')
ax1.set_ylabel('Gain Error - PPM')
ax1.legend(loc='upper left')

# Secondary y-axis for avg_temp (deduplicated by timestamp).
env_df = df.sort_values('timestamp').drop_duplicates(subset=['timestamp'])

ax2 = ax1.twinx()
ax2.plot(env_df['timestamp'], env_df['avg_temp'], 'r-', label='Avg Chamber Temp', linewidth=2)
ax2.set_ylabel('Avg Chamber Temperature (°C)', color='r')
ax2.tick_params(axis='y', labelcolor='r')

ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 60))
ax3.plot(env_df['timestamp'], env_df['avg_rh'], 'g-', label='Avg Chamber RH%', linewidth=2)
ax3.set_ylabel('Avg Chamber RH (%)', color='g')
ax3.tick_params(axis='y', labelcolor='g')

plt.title('PXIe-4080 Gain Error (10V and 300V) and Chamber Conditions Over Time')
fig.tight_layout()
plt.savefig(OUTPUT_IMAGE, dpi=300)
plt.show()